# ACTIVIDAD 2: REDES NEURONALES CONVOLUCIONALES

---

En esta actividad, vamos a trabajar con Convolutional Neural Networks para resolver un problema de clasificación de imágenes. En particular, vamos a clasificar diez clases que incluyen fundamentalmente animales y vehículos.

Como las CNN profundas son un tipo de modelo bastante avanzado y computacionalmente costoso, se recomienda hacer la práctica en Google Colaboratory con soporte para GPUs. En [este enlace](https://medium.com/deep-learning-turkey/google-colab-free-gpu-tutorial-e113627b9f5d) se explica cómo activar un entorno con GPUs. *Nota: para leer las imágenes y estandarizarlas al mismo tamaño se usa la librería opencv. Esta ĺibrería está ya instalada en el entorno de Colab, pero si trabajáis de manera local tendréis que instalarla.*

<center><img src="https://production-media.paperswithcode.com/datasets/4fdf2b82-2bc3-4f97-ba51-400322b228b1.png" style="text-align: center" height="300px"></center>

El dataset a utilizar consiste en 60000 imágenes a color de 10 clases de animales y vehículos. El dataset en cuestión se denomina [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) y es más complejo que el dataset MNIST que hemos utilizado en la actividad 1. Aunque tiene las mismas clases (10), los animales y vehículos pueden aparecer en distintas poses, en distintas posiciones de la imagen o con otros animales/ vehículos en pantalla (si bien el elemento a clasificar siempre aparece en la posición predominante).

## Carga de los datos

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix

# Fijamos semillas para que las comparaciones sean reproducibles.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [2]:
# Primero, definimos los datos de entrenamiento, validación y prueba
(X, Y), (x_test, y_test) = cifar10.load_data()
(x_train, x_valid) = (X[:40000], X[40000:])
(y_train, y_valid) = (Y[:40000], Y[40000:])

NameError: name 'cifar10' is not defined

In [4]:
# Normalizamos y convertimos a float32 para reducir memoria y acelerar el entrenamiento.
x_train = x_train.astype("float32") / 255.0
x_valid = x_valid.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


In [ ]:
# Esta variable contiene un mapeo de número de clase a elemento (animal o vehículo).
# La incluimos para ayudarte con la identificación de clases. De ti depende
# si quieres utilizar esta variable o no
MAP_ELEMENTS = {
    0: 'avion', 1: 'coche', 2: 'ave',
    3: 'gato', 4: 'ciervo', 5: 'perro', 6: 'rana',
    7: 'caballo', 8: 'barco', 9: 'camion'
}

In [7]:
# Función auxiliar para convertir las etiquetas a codificación one-hot
def convert_to_one_hot(labels, num_classes):
    return np.squeeze(np.array([to_categorical(label, num_classes=num_classes) for label in labels]))

# Convertimos las etiquetas de entrenamiento, validación y prueba
num_classes = 10
y_train_one_hot = convert_to_one_hot(y_train, num_classes)
y_valid_one_hot = convert_to_one_hot(y_valid, num_classes)
y_test_one_hot = convert_to_one_hot(y_test, num_classes)

# Verificamos las conversiones
print(y_train_one_hot.shape)
print(y_valid_one_hot.shape)
print(y_test_one_hot.shape)

(40000, 10)
(10000, 10)
(10000, 10)


## Ejercicio

Utilizando Convolutional Neural Networks con Keras, entrenar un clasificador que sea capaz de reconocer una imagen de las incluidas en CIFAR-10 con la mayor accuracy posible. Redactar un informe analizando varias de las alternativas probadas y los resultados obtenidos.

A continuación se detallan una serie de aspectos orientativos que podrían ser analizados en vuestro informe (no es necesario tratar todos ellos ni mucho menos, esto son ideas orientativas de aspectos que podéis explorar):

*   Análisis de los datos a utilizar.
*   Análisis de resultados, obtención de métricas de *precision* y *recall* por clase y análisis de qué clases obtienen mejores o peores resultados.
*   Análisis visual de los errores de la red. ¿Qué tipo de imágenes dan más problemas a nuestro modelo?
*   Comparación de modelos CNNs con un modelo de Fully Connected para este problema.
*   Utilización de distintas arquitecturas CNNs, comentando aspectos como su profundidad, hiperparámetros utilizados, optimizador, uso de técnicas de regularización, *batch normalization*, etc.
*   [ *algo más difícil* ] Utilización de *data augmentation*. Esto puede conseguirse con la clase [ImageDataGenerator](https://keras.io/preprocessing/image/#imagedatagenerator-class) de Keras.

Notas:
* Te recomendamos mantener los conjuntos de entrenamiento, test y prueba que se crean en el Notebook. No obstante, si crees que modificando tales conjuntos puedes lograr mejores resultados (o que puedes lograr los mismos resultados con menos datos, lo cual es también un logro), eres libre de hacerlo.
* No es necesario mostrar en el notebook las trazas de entrenamiento de todos los modelos entrenados, si bien una buena idea seria guardar gráficas de esos entrenamientos para el análisis. Sin embargo, **se debe mostrar el entrenamiento completo del mejor modelo obtenido y la evaluación de los datos de test con este modelo**.

### Exploración inicial

Comprobamos el tamaño de los conjuntos y la distribución de clases antes de entrenar. El conjunto de test se reserva para la evaluación final.


In [ ]:
# Distribución de clases en entrenamiento y validación
class_names = [MAP_ELEMENTS[i] for i in range(num_classes)]
train_counts = np.bincount(y_train.reshape(-1), minlength=num_classes)
valid_counts = np.bincount(y_valid.reshape(-1), minlength=num_classes)

print(f"Entrenamiento: {x_train.shape}, validación: {x_valid.shape}, test: {x_test.shape}")
class_distribution = pd.DataFrame({
    "clase": class_names,
    "entrenamiento": train_counts,
    "validación": valid_counts,
})
display(class_distribution)

fig, ax = plt.subplots(figsize=(10, 4))
class_distribution.set_index("clase").plot(kind="bar", ax=ax)
ax.set_ylabel("Número de imágenes")
ax.set_title("Distribución de clases")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


### Modelos candidatos

Comparamos un baseline *fully connected*, una CNN sencilla y una CNN más profunda con normalización por lotes, *dropout* y aumento de datos. Todos usan la misma partición de entrenamiento/validación; la selección se hace por `val_accuracy`.


In [ ]:
from tensorflow.keras import layers

def compile_model(model):
    """Compila un clasificador multiclase con el mismo criterio de comparación."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return model


def build_fully_connected():
    """Baseline denso: permite comparar el efecto de preservar la estructura espacial."""
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ], name="fully_connected")
    return compile_model(model)


def build_small_cnn():
    """CNN compacta sin aumento de datos, como primer modelo convolucional."""
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.35),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_sencilla")
    return compile_model(model)


def build_deep_cnn():
    """CNN profunda con aumento de datos, BatchNorm y regularización."""
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ], name="aumento_datos")

    inputs = keras.Input(shape=(32, 32, 3))
    x = augmentation(inputs)
    for filters, dropout_rate in [(64, 0.2), (128, 0.3), (256, 0.4)]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return compile_model(keras.Model(inputs, outputs, name="cnn_profunda"))


model_builders = {
    "Fully connected": build_fully_connected,
    "CNN sencilla": build_small_cnn,
    "CNN profunda + augmentation": build_deep_cnn,
}


### Entrenamiento y selección del mejor modelo

Se entrena cada alternativa con *early stopping* y reducción adaptativa del *learning rate*. El mejor checkpoint se determina únicamente con validación. Puede llevar bastante tiempo, especialmente en CPU.


In [ ]:
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints_cifar10")
CHECKPOINT_DIR.mkdir(exist_ok=True)
EPOCHS = 30
BATCH_SIZE = 128
histories = {}
validation_scores = {}

for model_name, build_model in model_builders.items():
    print(f"\nEntrenando: {model_name}")
    model = build_model()
    checkpoint_path = CHECKPOINT_DIR / f"{model.name}.keras"
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path), monitor="val_accuracy",
            mode="max", save_best_only=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3,
            min_lr=1e-6, verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=7,
            restore_best_weights=True, verbose=1,
        ),
    ]
    history = model.fit(
        x_train, y_train_one_hot,
        validation_data=(x_valid, y_valid_one_hot),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1,
    )
    histories[model_name] = history.history
    validation_scores[model_name] = max(history.history["val_accuracy"])

results = pd.DataFrame.from_dict(
    validation_scores, orient="index", columns=["mejor_accuracy_validación"]
).sort_values("mejor_accuracy_validación", ascending=False)
display(results)

best_model_name = results.index[0]
# Cargamos el checkpoint con mayor val_accuracy para garantizar que se evalúa
# la mejor época y no necesariamente la última.
model_name_to_file = {
    "Fully connected": "fully_connected.keras",
    "CNN sencilla": "cnn_sencilla.keras",
    "CNN profunda + augmentation": "cnn_profunda.keras",
}
best_model = keras.models.load_model(CHECKPOINT_DIR / model_name_to_file[best_model_name])
print(f"Modelo seleccionado: {best_model_name} (val_accuracy={results.iloc[0, 0]:.4f})")


## Análisis de datos y comparación de modelos

### Datos y preparación

CIFAR-10 contiene 60.000 imágenes RGB de 32 × 32 píxeles distribuidas en 10 clases equilibradas. En este notebook se usan 40.000 imágenes para entrenar, 10.000 para validación y las 10.000 del conjunto oficial de test para la evaluación final. Las imágenes se escalan a `[0, 1]` y las etiquetas se convierten a codificación *one-hot*. La partición de validación permite escoger el modelo sin usar el conjunto de test durante el ajuste.

La resolución reducida hace que algunos objetos ocupen pocos píxeles y que clases visualmente parecidas puedan confundirse. Por eso, además de la accuracy global, conviene revisar precisión y *recall* por clase, la matriz de confusión y ejemplos de errores.

### Resultados de entrenamiento y validación

La tabla resume el registro recibido. La accuracy de entrenamiento corresponde a la época en la que cada modelo alcanzó su mejor `val_accuracy`.

| Modelo | época con mejor `val_accuracy` | Accuracy de entrenamiento | Mejor accuracy de validación | `val_loss` en esa época |
|---|---:|---:|---:|---:|
| Fully connected | 24 | 35,66 % | 40,02 % | 1,8177 |
| CNN sencilla | 30 | 85,25 % | 80,10 % | 0,8217 |
| CNN profunda + *augmentation* | 30 | 85,37 % | **84,11 %** | **0,7185** |

En el baseline denso, la accuracy de validación (40,02 %) es superior a la accuracy de entrenamiento registrada en su mejor época (35,66 %). Esto puede ocurrir porque el *dropout* está activo durante el entrenamiento y desactivado al validar; no debe interpretarse por sí solo como fuga de datos.

La CNN sencilla supera al modelo *fully connected* en 40,08 puntos porcentuales de accuracy de validación. La CNN profunda mejora a la sencilla en otros 4,01 puntos y obtiene una pérdida de validación menor. Por tanto, de estas tres alternativas, la CNN profunda con aumento de datos es la mejor según validación.

### Comparación de arquitecturas e hiperparámetros

- **Fully connected:** aplana cada imagen y la procesa con capas densas de 512 y 256 neuronas, con *dropout* de 0,4 y 0,3. Su 40,02 % de accuracy de validación muestra que, en esta tarea, no aprovecha tan bien la estructura espacial. Al aplanar, se pierde la vecindad entre píxeles y el modelo debe aprender desde cero relaciones que las convoluciones representan de forma más directa.
- **CNN sencilla:** usa dos bloques convolucionales (32 y 64 filtros), *max pooling*, *dropout* y una capa densa de 128 neuronas. Alcanza 80,10 % de validación, una mejora grande respecto al baseline denso. La accuracy de entrenamiento al final fue 85,25 %, unos 5,15 puntos por encima de validación: existe una brecha compatible con cierto sobreajuste.
- **CNN profunda + augmentation:** utiliza bloques de 64, 128 y 256 filtros, cada uno con convoluciones, *batch normalization*, *max pooling* y *dropout*, seguidos de *global average pooling*. La accuracy de entrenamiento y validación al final fueron 85,37 % y 84,11 %; la brecha de 1,26 puntos es menor que en la CNN sencilla. El resultado es consistente con mejor generalización en esta ejecución.

Los tres modelos comparten Adam con tasa inicial `0.001`, entropía cruzada categórica con *label smoothing* `0.05`, lotes de 128 y un máximo de 30 épocas. `ReduceLROnPlateau` redujo la tasa de aprendizaje a `0.0005` para la CNN sencilla y a `0.0000625` para la profunda; la tasa del modelo denso se mantuvo en `0.001`. El ajuste automático de la tasa ayuda cuando la pérdida de validación deja de mejorar. El *checkpoint* conserva los pesos con mejor `val_accuracy`; para el modelo denso fue la época 24, mientras que las CNN alcanzaron su mejor resultado en la época 30.

### Regularización y aumento de datos

La CNN profunda combina *dropout*, *batch normalization* y transformaciones aleatorias de las imágenes de entrenamiento: volteo horizontal, pequeños desplazamientos, rotación y zoom. El aumento de datos presenta variaciones plausibles de cada imagen y puede ayudar a que el modelo no dependa de una posición o apariencia exacta. En estos resultados, la CNN profunda rinde mejor y muestra una brecha menor entre entrenamiento y validación.

No se puede atribuir toda la mejora únicamente al *augmentation* o a *batch normalization*: frente a la CNN sencilla también cambian la profundidad, el número de filtros, la regularización y la estrategia de *pooling*. Para medir el efecto independiente de cada técnica haría falta una comparación controlada, modificando una sola decisión cada vez.

### Evaluación final en el conjunto de test

La CNN profunda con *augmentation*, seleccionada mediante validación, obtiene **83,63 % de accuracy en test** (8.363 aciertos de 10.000; 1.637 errores). La diferencia respecto al 84,11 % de validación es de **0,48 puntos porcentuales**, por lo que ambas medidas son cercanas en esta ejecución. Este resultado respalda la capacidad de generalización del modelo, aunque el rendimiento sigue variando claramente entre clases.

| Clase | Precisión | Recall | F1 |
|---|---:|---:|---:|
| avión | 85,91 % | 87,20 % | 86,55 % |
| coche | 91,69 % | 93,80 % | **92,73 %** |
| ave | 84,96 % | 77,40 % | 81,00 % |
| gato | 78,97 % | **63,10 %** | **70,15 %** |
| ciervo | 86,28 % | 78,00 % | 81,93 % |
| perro | 85,46 % | 67,00 % | 75,11 % |
| rana | 73,07 % | 94,40 % | 82,37 % |
| caballo | 85,38 % | 88,80 % | 87,06 % |
| barco | **91,89 %** | 90,60 % | 91,24 % |
| camión | 77,05 % | **96,00 %** | 85,49 % |

La precisión macro y ponderada es 84,07 %, el *recall* es 83,63 % y el F1 es 83,36 %. Cada clase tiene 1.000 imágenes de test, así que los promedios macro y ponderado coinciden (salvo redondeos). El *recall* global coincide con la accuracy porque el conjunto está equilibrado.

### Lectura de la matriz de confusión y errores

La matriz está normalizada por etiqueta verdadera; cada fila indica cómo se distribuyen las imágenes reales de esa clase entre las predicciones. Su diagonal coincide con el *recall* de cada clase. Los patrones más claros son:

- **Gato y perro:** son las clases con menor *recall* (63,10 % y 67,00 %). El 10 % de los gatos se predice como rana y el 7 % como perro. Entre los perros, el 10 % se predice como gato y el 7 % como rana. Esto apunta a confusiones entre clases de apariencia similar en estas imágenes pequeñas.
- **Ave y ciervo:** el 8 % de cada clase se clasifica como rana; el ciervo también se confunde con caballo (5 %). Son errores visibles en las filas de la matriz y explican parte de sus *recalls* más bajos.
- **Rana y camión:** tienen *recall* muy alto (94,40 % y 96,00 %), pero precisión menor (73,07 % y 77,05 %). El modelo encuentra casi todas las ranas y camiones reales, pero también asigna esas etiquetas a imágenes de otras clases. En la matriz se observan varias clases que se predicen como rana y, para camión, falsos positivos desde avión, coche, gato y barco, entre otras.
- **Coche y barco:** obtienen los F1 más altos (92,73 % y 91,24 %), con precisión y *recall* superiores al 90 %. El modelo separa estas clases con más fiabilidad en este conjunto.

Las **1.637 imágenes mal clasificadas** representan el 16,37 % del test y son coherentes con la accuracy de 83,63 %. La galería de errores permite complementar la matriz: sirve para comprobar si las confusiones corresponden a objetos pequeños, poses poco habituales, fondos complejos o imágenes ambiguas; la matriz por sí sola no permite atribuir cada error a esas causas.

### Conclusión final

Entre los tres candidatos, el modelo *fully connected* quedó muy por debajo de las CNN (40,02 % de accuracy de validación frente a 80,10 % y 84,11 %). Esto es consistente con la ventaja de conservar la estructura espacial mediante convoluciones. La CNN profunda con *augmentation* fue el mejor candidato según validación y obtuvo **83,63 % de accuracy en test**, con un F1 macro de **83,36 %**.

El resultado por clase muestra que la accuracy global oculta diferencias importantes: coche y barco se reconocen especialmente bien, mientras que gato y perro concentran los *recalls* más bajos. Rana y camión muestran el patrón opuesto: *recall* muy alto y precisión más baja, señal de que el modelo predice esas clases con demasiada frecuencia. Como siguientes experimentos, convendría inspeccionar esas confusiones y probar cambios controlados en el aumento de datos o en la arquitectura. La comparación actual cambia varias decisiones a la vez, así que no permite aislar cuál de ellas produjo la mejora.

La evaluación de test debe quedar como medida final del experimento; cualquier nuevo ajuste debería seleccionarse con validación para no adaptar los modelos a estos 10.000 ejemplos.

### Evolución del entrenamiento

Mostramos las curvas completas del modelo seleccionado, incluyendo las épocas detenidas por *early stopping*.


In [3]:
best_history = histories[best_model_name]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(best_history["accuracy"], label="entrenamiento")
axes[0].plot(best_history["val_accuracy"], label="validación")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("época")
axes[0].legend()
axes[1].plot(best_history["loss"], label="entrenamiento")
axes[1].plot(best_history["val_loss"], label="validación")
axes[1].set_title("Loss")
axes[1].set_xlabel("época")
axes[1].legend()
plt.tight_layout()
plt.show()


NameError: name 'histories' is not defined

### Evaluación final sobre test

El conjunto de test se utiliza una sola vez, después de seleccionar el modelo mediante validación.


In [ ]:
test_loss, test_accuracy = best_model.evaluate(x_test, y_test_one_hot, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = y_test.reshape(-1)
y_pred = np.argmax(best_model.predict(x_test, batch_size=BATCH_SIZE, verbose=1), axis=1)
print(classification_report(
    y_true, y_pred, target_names=class_names, digits=4, zero_division=0
))


In [ ]:
# Matriz de confusión normalizada por clase verdadera
cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(num_classes), yticks=range(num_classes),
       xticklabels=class_names, yticklabels=class_names,
       xlabel="Predicción", ylabel="Etiqueta real",
       title="Matriz de confusión normalizada (test)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
for row in range(num_classes):
    for col in range(num_classes):
        ax.text(col, row, f"{cm[row, col]:.2f}", ha="center", va="center",
                color="white" if cm[row, col] > 0.55 else "black", fontsize=8)
fig.colorbar(image, ax=ax, label="Proporción")
plt.tight_layout()
plt.show()


### Inspección visual de errores

Las siguientes imágenes permiten observar qué ejemplos confunde el modelo y entre qué clases.


In [ ]:
wrong = np.flatnonzero(y_true != y_pred)
print(f"Imágenes mal clasificadas: {len(wrong)} de {len(y_true)}")

if len(wrong):
    shown = wrong[:15]
    cols = 5
    rows = int(np.ceil(len(shown) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 2.7 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, idx in zip(axes, shown):
        ax.imshow(x_test[idx])
        ax.set_title(f"Real: {class_names[y_true[idx]]}\nPred.: {class_names[y_pred[idx]]}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(shown):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()